# Session 4 — Bandits: an RL-based agent

**Salih Tutun, PhD** · DAT 5668 · Module 2 · Practice session (about 80 minutes)**

I am Salih Tutun. We leave documents. I talk. You guess to a neighbor. Until I say **open the notebook**, this file stays closed. **Do not scroll.** One cell at a time. Two voices, then type. After we run: **LLM**, **RL**, or **hybrid** — and one line for your project.

**Family this session: RL-based agent.** Repeated numeric choices and a reward. Session 3's LLM-based agent does not fit this job, and this bandit does not fit Session 3's documents.

![A bandit picks one FX sleeve](https://raw.githubusercontent.com/salihtutun/Agentic-AI_Autonomous_Decision_Making/main/assets/desk-bandit.png)

## Run of show
- 10 min — Why this is not Session 3's document problem.
- 55 min — Implement ε-greedy and read the regret.
- 15 min — When a bandit is the wrong tool (and an LLM agent is the wrong tool).

## Business setting
A treasury desk parks overnight cash in **one** FX sleeve: EUR, GBP, or JPY (versus USD). Only the sleeve you chose pays you that day. You never see the return of the sleeves you skipped. That is a multi-armed bandit.

The rates are **real**: daily USD quotes from [Frankfurter](https://www.frankfurter.app/) (public-apis, No Auth), 2024 Q1. The agent does not get the mean returns in advance.

## The rule, before you code it

Each business day the agent picks one sleeve. The reward is that day's return (it can be negative). You never observe the sleeves you did not pick.

**ε-greedy.** Draw a number uniformly between 0 and 1. If it is below `epsilon`, return a random sleeve index. Otherwise return the smallest index whose value is largest. `epsilon = 0` never explores. `epsilon = 1` never uses what it has learned. `0.10` spends most days on the current best and still checks the others.

**Sample mean.** After reward `r` on sleeve index `a`:

- `counts[a] = counts[a] + 1`
- `values[a] = values[a] + (r - values[a]) / counts[a]`

That is the average of the rewards seen for that sleeve, written so you do not store every day. The first term in the parentheses is the surprise. Dividing by the new count shrinks the step as evidence piles up.

**Regret** is the extra return the best sleeve would have earned, minus what you actually got. A low regret means the explore/exploit balance worked. A run that locks onto EUR because GBP was unlucky in week one has high regret even if the code is correct.


## How we work in this notebook

I am Salih. I talk. You guess to a neighbor. Two voices share. Then we type **this cell only**. Do not scroll.

| You will see | What you do in class |
| --- | --- |
| **STOP** | Wait. Do not run the next cell until I say. |
| **This cell — ...** | I explain. Then we run it together. |
| **YOU FILL THIS** | Guess to a neighbor. Two people share. Then type `YOU TYPE HERE`. |
| **TODO** / `YOU WRITE THIS` | You write the function. Search for `TODO`. We stay until every line says PASS. |
| **RUN WITH ME** | Already written. Run it. Do not type in that cell. |
| **Family check** | Say **LLM**, **RL**, or **hybrid** — and one line for your project framework. |


In [ ]:
# RUN WITH ME. Real Frankfurter USD rates, then daily returns. No API key.
from pathlib import Path
import csv
import numpy as np

def course_data(name):
    for base in (Path("data"), Path("../../assets/data"), Path("assets/data"), Path("/content/data")):
        path = base / name
        if path.exists():
            return path
    raise FileNotFoundError(name)


def load_fx_returns():
    """Overnight return of each sleeve. Only the chosen sleeve is observed in the loop."""
    rows = list(csv.DictReader(course_data("fx-usd-sleeves.csv").open(encoding="utf-8")))
    levels = {name: [float(row[name]) for row in rows] for name in ("EUR", "GBP", "JPY")}
    returns = {}
    for name, series in levels.items():
        returns[name] = [(series[i] - series[i - 1]) / series[i - 1] for i in range(1, len(series))]
    return returns


OFFERS = ["EUR", "GBP", "JPY"]
RETURNS = load_fx_returns()
N_DAYS = len(RETURNS["EUR"])
TRUE_MEAN = {name: float(np.mean(RETURNS[name])) for name in OFFERS}
RNG = np.random.default_rng(7)


def sleeve_reward(offer, day):
    """One business day. Reward is that sleeve's real return. The others stay hidden."""
    return RETURNS[offer][day % N_DAYS]


def best_possible_rate():
    return max(TRUE_MEAN.values())


print("Sleeves", OFFERS, "days", N_DAYS)
print("Mean daily return (agent does not get this table):", {k: round(v, 5) for k, v in TRUE_MEAN.items()})
print("Best mean sleeve:", max(TRUE_MEAN, key=TRUE_MEAN.get))


## Your work — TODO (you write this in class)

Search for `TODO`.

1. Complete `epsilon_greedy`. With probability `epsilon`, pick a random offer index. Otherwise pick an index of the largest value in `values` (break ties toward the smallest index).
2. Complete the loop body where marked so each visit updates `counts` and `values` with the sample mean.
3. Leave `epsilon` at 0.10, then rerun the last cell with 0.0 and with 0.40 and write one sentence on what changed.


In [ ]:
def epsilon_greedy(values, epsilon, rng):
    """TODO: return an action index (0, 1, or 2)."""
    # TODO
    return 0


def run_bandit(epsilon, n_visits=None):
    n_visits = n_visits or N_DAYS
    values = np.zeros(len(OFFERS))
    counts = np.zeros(len(OFFERS))
    rewards = []
    for day in range(n_visits):
        index = epsilon_greedy(values, epsilon, RNG)
        offer = OFFERS[index]
        reward = sleeve_reward(offer, day)
        # TODO: update counts[index] and values[index] to the new sample mean.
        rewards.append(reward)
    return values, counts, np.cumsum(rewards)


values, counts, cumulative = run_bandit(0.10)
print("values", np.round(values, 5))
print("counts", counts)
print("cumulative return", round(float(cumulative[-1]), 5) if len(cumulative) else 0)


In [ ]:
def regret_so_far(cumulative):
    """Extra return the best mean sleeve would have collected."""
    visits = np.arange(1, len(cumulative) + 1)
    best = best_possible_rate() * visits
    return best - cumulative


def report(epsilon):
    values, counts, cumulative = run_bandit(epsilon)
    if counts.sum() == 0:
        print(f"epsilon={epsilon}: NOT YET — the loop never updated the counts")
        return
    gap = regret_so_far(cumulative)[-1]
    learned = OFFERS[int(np.argmax(values))]
    print(
        f"epsilon={epsilon}: learned best={learned}  counts={counts.astype(int).tolist()}  "
        f"regret vs best mean sleeve {gap:.4f}"
    )
    print("  A finished run should lean toward the sleeve with the best 2024 Q1 mean, and still sample the others when epsilon is 0.10.")


for epsilon in (0.0, 0.10, 0.40):
    report(epsilon)


## Debrief

1. What goes wrong at epsilon 0 if the first unlucky week makes GBP look worse than EUR?
2. Why is a CFPB billing dispute a poor job for this bandit?
3. Name a business test you would **not** run as a bandit, because the cost of one bad pull is too high (a large wire, a medical denial, an auto-refund).


## Fill this with me — type this in class

**STOP. Do not scroll.** Neighbor 20s. Two voices. After `epsilon_greedy` works, pick an epsilon as a class. 0 never explores. 1 never settles. 0.10 is the useful middle.


In [ ]:
# FILL WITH THE CLASS. >>> YOU FILL THIS <<<
# Neighbor first. A number from 0 to 1.
class_epsilon = None  # YOU TYPE HERE, for example 0.10
if class_epsilon is None:
    print("Agree on an epsilon, type it, and run again.")
else:
    report(class_epsilon)
    print("Say whether this was too shy or too restless.")


## Add this to your framework — STOP. Do not scroll.

Guess to a neighbor (20 seconds): after today, is **your** project **LLM**, **RL**, or **hybrid**? Two voices share. Then type one sentence you will keep.

Hint: an overnight FX sleeve is **RL**. Session 3's CFPB complaint is **LLM**. Hybrid later: learned sleeve on the usual day; a language agent on the exception email.


In [ ]:
# FILL WITH THE CLASS. >>> YOU FILL THIS <<<
# Neighbor first. Two voices. Then type.
framework_family = ""  # YOU TYPE HERE: LLM, RL, or hybrid
framework_line = ""    # YOU TYPE HERE: one sentence you will keep
print("Family:", framework_family or "(say it, then type it)")
print("I will keep:", framework_line or "(one sentence)")
